# Lab: Adaptation Decisions

**Module 01 — Architecture Fundamentals (Session 03)**

## Objectives

By the end of this lab you will be able to:

1. **Classify** real product asks against the slide's adaptation ladder: Prompting → RAG → Fine-tuning → Train.
2. **Defend** your classification against the slide's four legitimate reasons to fine-tune.
3. **Recognise** the anti-patterns (e.g., fine-tuning to teach facts that change quarterly).
4. **Disagree with an LLM** when its classification differs from yours — and articulate why.

## Why this lab is mostly markdown

The hardest skill in this session isn't writing code — it's **resisting** the urge to fine-tune. Five scenarios. For each:

1. Read the prompt.
2. Write your one-paragraph classification + justification in the markdown cell that follows.
3. Read the inline rubric to see what we'd argue. If you disagree, write down *why* — that's the point of the exercise.

A final code cell asks the LLM judge from the shared `eval_kit` to classify the same scenarios. You can — and should — disagree with it.


## Setup


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / 'shared'))

from eval_kit.judge import LLMJudge
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("OPENROUTER_API_KEY"), (
    "Set OPENROUTER_API_KEY in .env"
)
print("ready")


---
## Recap — the adaptation ladder

From the slides:

| Rung | Cost / time | When |
|---|---|---|
| **Prompting** | hours | Default. Try this first. Three+ rounds of failed iteration → consider the next rung. |
| **RAG** | days | Knowledge the model doesn't have, or that changes faster than retraining. |
| **Fine-tuning** | weeks | One of the four legitimate reasons holds AND you have ≥500 worked examples AND you have an eval. |
| **Train from scratch** | months / $$$ | You are a foundation-model lab. You aren't. |

**Four legitimate reasons to fine-tune** (slide verbatim):

1. Format / style consistency at scale.
2. Domain jargon and dialect.
3. Implicit reasoning patterns.
4. Latency / cost compression.

**The Galactica cautionary tale:** fine-tuning teaches **form**, not **truth**. A model that confidently fabricates in the right register is more dangerous than one that obviously fails.


---
### Scenario 1 — The product catalog that keeps changing

> Your e-commerce team wants the chatbot to answer questions about your 2026 product catalog. The catalog is updated **every quarter** (new SKUs, retired items, price changes).

**Your turn.** In the cell below, write your classification (Prompting / RAG / Fine-tuning / Train) and one paragraph of justification. Reference the four legitimate reasons (or anti-patterns) where they apply.


*Your classification + justification:*

> _Replace this italic line with your answer._


::: {.callout-tip collapse="true"}
### Our rubric — open after you've written your own answer

**Recommended:** RAG

Knowledge changes faster than any reasonable fine-tune cycle. Fine-tuning bakes in stale facts and turns price changes into hallucination risk. RAG lets you re-index on every catalog update without touching the model.
:::


---
### Scenario 2 — A non-standard SQL dialect

> Your internal data warehouse uses **custom UDFs and non-standard date functions** (`date_trunc_fiscal`, `to_riyadh_tz`). Analysts paste questions in chat and want syntactically correct SQL back.

**Your turn.** In the cell below, write your classification (Prompting / RAG / Fine-tuning / Train) and one paragraph of justification. Reference the four legitimate reasons (or anti-patterns) where they apply.


*Your classification + justification:*

> _Replace this italic line with your answer._


::: {.callout-tip collapse="true"}
### Our rubric — open after you've written your own answer

**Recommended:** Fine-tuning

Pure format/style task: the syntax is finite, demonstrable with 500–2000 worked examples, and stable across releases. Fits the slide's first legitimate reason (format consistency at scale). Prompting works for 80% but you'll never close the long tail. RAG is the wrong tool — you're not adding *facts*, you're shaping output.
:::


---
### Scenario 3 — Medical dosage refusal

> The assistant must **refuse to answer medical-dosage questions** but stay helpful on general health topics. Compliance will sign off on a refusal template.

**Your turn.** In the cell below, write your classification (Prompting / RAG / Fine-tuning / Train) and one paragraph of justification. Reference the four legitimate reasons (or anti-patterns) where they apply.


*Your classification + justification:*

> _Replace this italic line with your answer._


::: {.callout-tip collapse="true"}
### Our rubric — open after you've written your own answer

**Recommended:** Prompting (with guardrails)

A single sharp behavior change, defined by a template. Goes in the system prompt + Module 05 input/output filters. Fine-tuning here is overkill and harder to audit — when legal asks 'show me where the refusal is enforced', a prompt + filter is auditable; weights are not.
:::


---
### Scenario 4 — Internal jargon translation

> Support engineers want to translate English customer tickets into your **internal jargon** (~800 mapped terms: 'box' → 'shipment-1u', 'login' → 'auth-flow-v2'). The vocabulary is stable; you have a clean dictionary.

**Your turn.** In the cell below, write your classification (Prompting / RAG / Fine-tuning / Train) and one paragraph of justification. Reference the four legitimate reasons (or anti-patterns) where they apply.


*Your classification + justification:*

> _Replace this italic line with your answer._


::: {.callout-tip collapse="true"}
### Our rubric — open after you've written your own answer

**Recommended:** Prompting (with a dictionary lookup) — *or* Fine-tuning if quality lags

Start with prompting: paste the dictionary in the system prompt (or look up only the terms present in the ticket). Cheap, debuggable, you can swap dictionaries in minutes. Reach for fine-tuning only if the model keeps over-translating context or missing implicit usages — that's the slide's third legitimate reason (implicit reasoning patterns).
:::


---
### Scenario 5 — House-style contract summaries

> A law firm wants 3-bullet summaries of 50-page contracts that match their **partner's house style** (specific tone, ordering of points, mandatory boilerplate).

**Your turn.** In the cell below, write your classification (Prompting / RAG / Fine-tuning / Train) and one paragraph of justification. Reference the four legitimate reasons (or anti-patterns) where they apply.


*Your classification + justification:*

> _Replace this italic line with your answer._


::: {.callout-tip collapse="true"}
### Our rubric — open after you've written your own answer

**Recommended:** RAG + Prompting (and *maybe* fine-tuning later)

The 50-page contract is *facts the model doesn't have* → that's RAG (or long-context). The 3-bullet house style is *output shape* → that's prompting with a strong example block. Don't fine-tune until prompting has been tuned for 3+ rounds and is still missing the style consistently. The slide's 30+ rounds rule applies here.
:::


---
## A second opinion from an LLM judge

The shared `eval_kit.judge.LLMJudge` we'll use repeatedly in later modules can also act as a debate partner here. **It is not the answer key.** When it disagrees with you, your job is to decide which of you is reasoning correctly — not to defer.


In [ ]:
classifier = LLMJudge(
    rubric=(
        "Classify the product ask on the adaptation ladder. "
        "Score 0 = Prompting, 1 = RAG, 2 = Fine-tuning, 3 = Train from scratch. "
        "In the reasoning, name which of the four legitimate reasons (if any) "
        "drives a Fine-tuning recommendation."
    ),
    scale_max=3,
)

LABELS = {0: "Prompting", 1: "RAG", 2: "Fine-tuning", 3: "Train from scratch"}

SCENARIOS = [
    "Your e-commerce team wants the chatbot to answer questions about your 2026 product catalog. The catalog is updated **every quarter** (new SKUs, retired items, price changes).",
    "Your internal data warehouse uses **custom UDFs and non-standard date functions** (`date_trunc_fiscal`, `to_riyadh_tz`). Analysts paste questions in chat and want syntactically correct SQL back.",
    "The assistant must **refuse to answer medical-dosage questions** but stay helpful on general health topics. Compliance will sign off on a refusal template.",
    "Support engineers want to translate English customer tickets into your **internal jargon** (~800 mapped terms: 'box' \u2192 'shipment-1u', 'login' \u2192 'auth-flow-v2'). The vocabulary is stable; you have a clean dictionary.",
    "A law firm wants 3-bullet summaries of 50-page contracts that match their **partner's house style** (specific tone, ordering of points, mandatory boilerplate).",
]

for i, prompt in enumerate(SCENARIOS, start=1):
    verdict = classifier.score(prompt)
    print(f"Scenario {i}: judge says {LABELS[verdict.score]}")
    print(f"  reasoning: {verdict.reasoning}\n")


---
## Reflection

### Where do you and the LLM disagree?

For each disagreement, write one line: who's right, and why? The discipline of articulating the disagreement is the point — it's the same discipline you'll need when reviewing a teammate's pull request that proposes fine-tuning.

### Wrap-up

> Fine-tuning is not the default. It's the answer when prompting has been tried three+ rounds, an eval exists, and one of the four legitimate reasons holds. Anything else is paying weeks of cost for what hours of prompt iteration would have delivered.
